In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')
import random
import os
import json
from datetime import datetime


In [2]:
import pandas as pd
import os

RUNS_PATH = "artifacts/runs.csv"

def log_experiment(row_dict):
    if os.path.exists(RUNS_PATH):
        df = pd.read_csv(RUNS_PATH)
        df = pd.concat([df, pd.DataFrame([row_dict])], ignore_index=True)
    else:
        df = pd.DataFrame([row_dict])
    df.to_csv(RUNS_PATH, index=False)

In [3]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


In [4]:
df = pd.read_csv("data/S12-hw-dataset.csv")

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df['date'].min(), df['date'].max())
print("Missing:\n", df.isna().sum())

plt.figure(figsize=(10,4))
plt.plot(df['date'], df['target'])
plt.title("Target time series")
plt.xlabel("Date")
plt.ylabel("Target")
plt.tight_layout()
plt.savefig("artifacts/figures/raw_series.png", dpi=150, bbox_inches='tight')
plt.close()

Shape: (4320, 2)
Date range: 2025-01-01 00:00:00 2025-06-29 23:00:00
Missing:
 date      0
target    0
dtype: int64


In [5]:
print(f"Среднее значение: {df['target'].mean():.2f}")
print(f"Стандартное отклонение: {df['target'].std():.2f}")
print(f"Минимум: {df['target'].min():.2f}")
print(f"Максимум: {df['target'].max():.2f}")

from scipy.stats import linregress
slope, _, _, _, _ = linregress(range(len(df)), df['target'])
print(f"Тренд: {'восходящий' if slope > 0 else 'нисходящий'} (slope={slope:.4f})")

from statsmodels.tsa.stattools import acf
acf_values = acf(df['target'].values, nlags=14, fft=True)
print(f"Автокорреляция lag-7: {acf_values[7]:.3f}")
print(f"Автокорреляция lag-14: {acf_values[14]:.3f}")

Среднее значение: 135.61
Стандартное отклонение: 21.38
Минимум: 69.10
Максимум: 210.10
Тренд: восходящий (slope=0.0102)
Автокорреляция lag-7: 0.353
Автокорреляция lag-14: 0.169


In [6]:
df['lag_1'] = df['target'].shift(1)
df['lag_7'] = df['target'].shift(7)
df['lag_14'] = df['target'].shift(14)

df['rolling_mean_7'] = df['target'].shift(1).rolling(7).mean()
df['rolling_std_7'] = df['target'].shift(1).rolling(7).std()

df['day_of_week'] = df['date'].dt.dayofweek

df = df.dropna().reset_index(drop=True)

In [7]:
n = len(df)
train_size = int(n * 0.7)
val_size = int(n * 0.15)

train = df.iloc[:train_size].reset_index(drop=True)
validation = df.iloc[train_size:train_size + val_size].reset_index(drop=True)  
test = df.iloc[train_size + val_size:].reset_index(drop=True)

print(f"\nTrain size: {len(train)} ({len(train)/n*100:.1f}%)")
print(f"Validation size: {len(validation)} ({len(validation)/n*100:.1f}%)") 
print(f"Test size: {len(test)} ({len(test)/n*100:.1f}%)")

print(f"Train date range: {train['date'].min()} - {train['date'].max()}")
print(f"Validation date range: {validation['date'].min()} - {validation['date'].max()}") 
print(f"Test date range: {test['date'].min()} - {test['date'].max()}")

plt.figure(figsize=(12,5))
plt.plot(train['date'], train['target'], label='train', linewidth=2)
plt.plot(validation['date'], validation['target'], label='validation', linewidth=2) 
plt.plot(test['date'], test['target'], label='test', linewidth=2)
plt.axvline(x=train['date'].max(), color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=validation['date'].max(), color='gray', linestyle='--', alpha=0.5)  
plt.legend()
plt.title("Temporal Split: Train / Validation / Test")
plt.xlabel("Date")
plt.ylabel("Target")
plt.tight_layout()
plt.savefig("artifacts/figures/series_split.png", dpi=150, bbox_inches='tight')
plt.close()


Train size: 3014 (70.0%)
Validation size: 645 (15.0%)
Test size: 647 (15.0%)
Train date range: 2025-01-01 14:00:00 - 2025-05-07 03:00:00
Validation date range: 2025-05-07 04:00:00 - 2025-06-03 00:00:00
Test date range: 2025-06-03 01:00:00 - 2025-06-29 23:00:00


=== Почему random split некорректен для временных рядов ===

   - При случайном разбиении модель может обучаться на данных из будущего
   - Это нарушает причинно-следственную связь во времени
   - Временные ряды имеют автокорреляцию (зависимость от предыдущих значений)
   - Random split разрушает эту структуру


Используем ТОЛЬКО temporal split для корректной валидации!

In [8]:
features = [
    'lag_1','lag_7','lag_14',
    'rolling_mean_7','rolling_std_7',
    'day_of_week'
]

scaler = StandardScaler()
scaler.fit(train[features])

X_train = scaler.transform(train[features])
X_validation = scaler.transform(validation[features])  
X_test = scaler.transform(test[features])

y_train = train['target'].values
y_validation = validation['target'].values 
y_test = test['target'].values

In [9]:
def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


In [10]:
# Эксперимент B1
print("=== Эксперимент B1: Naive Last ===")
b1_pred = validation['lag_1'].values  
b1_mae = mean_absolute_error(y_validation, b1_pred)  
b1_rmse = np.sqrt(mean_squared_error(y_validation, b1_pred))
b1_mape = calculate_mape(y_validation, b1_pred)

print(f"B1 MAE: {b1_mae:.4f}")
print(f"B1 RMSE: {b1_rmse:.4f}")
print(f"B1 MAPE: {b1_mape:.4f}%")

=== Эксперимент B1: Naive Last ===
B1 MAE: 6.4406
B1 RMSE: 8.2035
B1 MAPE: 4.3922%


In [11]:
# Эксперимент B2
print("=== Эксперимент B2: Moving Average ===")
b2_pred = validation['rolling_mean_7'].values  
b2_mae = mean_absolute_error(y_validation, b2_pred) 
b2_rmse = np.sqrt(mean_squared_error(y_validation, b2_pred))
b2_mape = calculate_mape(y_validation, b2_pred)

print(f"B2 MAE: {b2_mae:.4f}")
print(f"B2 RMSE: {b2_rmse:.4f}")
print(f"B2 MAPE: {b2_mape:.4f}%")

=== Эксперимент B2: Moving Average ===
B2 MAE: 12.7266
B2 RMSE: 15.2421
B2 MAPE: 8.8303%


In [12]:
print("=== Эксперимент B3: Ridge ===")
model_ridge = Ridge()
model_ridge.fit(X_train, y_train)

val_pred_ridge = model_ridge.predict(X_validation)  
b3_val_mae = mean_absolute_error(y_validation, val_pred_ridge) 
b3_val_rmse = np.sqrt(mean_squared_error(y_validation, val_pred_ridge))
b3_val_mape = calculate_mape(y_validation, val_pred_ridge)

print(f"B3 Val MAE: {b3_val_mae:.4f}")
print(f"B3 Val RMSE: {b3_val_rmse:.4f}")
print(f"B3 Val MAPE: {b3_val_mape:.4f}%")

test_pred_ridge = model_ridge.predict(X_test)
b3_test_mae = mean_absolute_error(y_test, test_pred_ridge)
b3_test_rmse = np.sqrt(mean_squared_error(y_test, test_pred_ridge))
b3_test_mape = calculate_mape(y_test, test_pred_ridge)

print(f"B3 Test MAE: {b3_test_mae:.4f}")
print(f"B3 Test RMSE: {b3_test_rmse:.4f}")
print(f"B3 Test MAPE: {b3_test_mape:.4f}%")

=== Эксперимент B3: Ridge ===
B3 Val MAE: 7.1644
B3 Val RMSE: 8.7180
B3 Val MAPE: 4.7765%
B3 Test MAE: 7.3257
B3 Test RMSE: 9.0340
B3 Test MAPE: 4.6913%


In [13]:
WINDOW_SIZE = 14
BATCH_SIZE = 32
EPOCHS = 10

class TimeSeriesDataset(Dataset):
    def __init__(self, series, window):
        self.X = []
        self.y = []
        for i in range(len(series) - window):
            self.X.append(series[i:i+window])
            self.y.append(series[i+window])
        
        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_series = train['target'].values
validation_series = validation['target'].values 
test_series = test['target'].values

train_ds = TimeSeriesDataset(train_series, WINDOW_SIZE)
validation_ds = TimeSeriesDataset(validation_series, WINDOW_SIZE) 
test_ds = TimeSeriesDataset(test_series, WINDOW_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
validation_loader = DataLoader(validation_ds, batch_size=BATCH_SIZE, shuffle=False)  
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [14]:
class GRUModel(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.gru = nn.GRU(1, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = x.unsqueeze(-1)
        out, _ = self.gru(x)
        out = out[:, -1, :]
        return self.fc(out).squeeze()

model_gru = GRUModel(hidden_size=32).to(device)
optimizer = torch.optim.Adam(model_gru.parameters(), lr=1e-3)
criterion = nn.L1Loss()

In [15]:
print("=== Эксперимент R1: GRU ===")
best_val = float('inf')
train_losses, val_losses = [], []
best_val_mae = float('inf')
best_val_rmse = float('inf')
best_val_mape = float('inf')

for epoch in range(EPOCHS):
    model_gru.train()
    train_loss = 0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        pred = model_gru(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    model_gru.eval()
    validation_loss = 0
    val_preds = []
    val_targets = []
       
    with torch.no_grad():
        for x, y in validation_loader:  
            x, y = x.to(device), y.to(device)
            pred = model_gru(x)
            validation_loss += criterion(pred, y).item()  
            val_preds.extend(pred.cpu().numpy())
            val_targets.extend(y.cpu().numpy())
    
    val_preds = np.array(val_preds)
    val_targets = np.array(val_targets)
    
    epoch_mae = mean_absolute_error(val_targets, val_preds)
    epoch_rmse = np.sqrt(mean_squared_error(val_targets, val_preds))
    epoch_mape = calculate_mape(val_targets, val_preds)
    
    train_losses.append(train_loss)
    val_losses.append(validation_loss)
    
    print(f"Epoch {epoch}: train={train_loss:.3f}, validation={validation_loss:.3f}, MAE={epoch_mae:.4f}")  
    
    if epoch_mae < best_val_mae:
        best_val_mae = epoch_mae
        best_val_rmse = epoch_rmse
        best_val_mape = epoch_mape
        torch.save(model_gru.state_dict(), "artifacts/best_gru.pt")

plt.figure(figsize=(10,5))
plt.plot(train_losses, label='train loss', linewidth=2)
plt.plot(val_losses, label='validation loss', linewidth=2) 
plt.legend()
plt.title("GRU Learning Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("artifacts/figures/gru_learning_curves.png", dpi=150, bbox_inches='tight')
plt.close()

=== Эксперимент R1: GRU ===
Epoch 0: train=11905.448, validation=2935.123, MAE=146.6368
Epoch 1: train=11558.032, validation=2869.410, MAE=143.3512
Epoch 2: train=11273.276, validation=2812.116, MAE=140.4866
Epoch 3: train=11004.038, validation=2752.570, MAE=137.5093
Epoch 4: train=10721.684, validation=2694.079, MAE=134.5847
Epoch 5: train=10450.874, validation=2637.083, MAE=131.7349
Epoch 6: train=10184.681, validation=2580.748, MAE=128.9182
Epoch 7: train=9920.904, validation=2524.813, MAE=126.1214
Epoch 8: train=9658.677, validation=2469.150, MAE=123.3382
Epoch 9: train=9397.541, validation=2413.684, MAE=120.5649


In [16]:
print("=== Оценка лучшей модели на Test ===")

if best_val_mae < b3_val_mae:
    print("Лучшая модель: GRU")
    model_gru.load_state_dict(torch.load("artifacts/best_gru.pt", weights_only=True))
    model_gru.eval()
    
    test_preds = []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            pred = model_gru(x)
            test_preds.extend(pred.cpu().numpy())
    
    best_test_pred = np.array(test_preds)
    min_len = min(len(best_test_pred), len(y_test))
    best_test_pred = best_test_pred[:min_len]
    y_test_eval = y_test[:min_len]
    
    test_mae = mean_absolute_error(y_test_eval, best_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test_eval, best_test_pred))
    test_mape = calculate_mape(y_test_eval, best_test_pred)
   
    best_model = "GRU"
else:
    print("Лучшая модель: Ridge")
    best_test_pred = test_pred_ridge
    test_mae = b3_test_mae
    test_rmse = b3_test_rmse
    test_mape = b3_test_mape
    best_model = "Ridge"

print(f"Test MAE: {test_mae:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAPE: {test_mape:.4f}%")

=== Оценка лучшей модели на Test ===
Лучшая модель: Ridge
Test MAE: 7.3257
Test RMSE: 9.0340
Test MAPE: 4.6913%


In [17]:
config = {
    "model": "GRU",
    "hidden_size": 32,
    "window_size": WINDOW_SIZE,
    "batch_size": BATCH_SIZE,
    "learning_rate": 1e-3,
    "epochs": EPOCHS,
    "seed": seed,
    "device": str(device),
    "optimizer": "Adam",
    "loss": "L1Loss",
    "features": "raw_sequence",
    "scaler": "None"
}

with open("artifacts/best_gru_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Конфигурация сохранена в artifacts/best_gru_config.json")

Конфигурация сохранена в artifacts/best_gru_config.json


In [18]:
plt.figure(figsize=(10,5))
models = ['B1\nNaive', 'B2\nMA', 'B3\nRidge', 'R1\nGRU']
maes = [b1_mae, b2_mae, b3_val_mae, best_val_mae]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4']

bars = plt.bar(models, maes, color=colors, edgecolor='black', linewidth=1.5)
plt.ylabel('MAE', fontsize=12)
plt.title('Baseline Comparison on Validation', fontsize=14)
plt.grid(axis='y', alpha=0.3)

# Добавляем значения на столбцы
for bar, ma in zip(bars, maes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{ma:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig("artifacts/figures/baselines_compare.png", dpi=150, bbox_inches='tight')
plt.close()

print("График сравнения baseline сохранен в artifacts/figures/baselines_compare.png")

График сравнения baseline сохранен в artifacts/figures/baselines_compare.png


In [19]:
plt.figure(figsize=(14,6))

# Для GRU нужно скорректировать даты (из-за window_size)
if best_model == "GRU":
    plot_dates = test['date'].values[WINDOW_SIZE:WINDOW_SIZE+len(best_test_pred)]
    plot_actual = y_test[WINDOW_SIZE:WINDOW_SIZE+len(best_test_pred)]
else:
    plot_dates = test['date'].values[:len(best_test_pred)]
    plot_actual = y_test[:len(best_test_pred)]

plt.plot(plot_dates, plot_actual, label='Actual', linewidth=2, alpha=0.8)
plt.plot(plot_dates, best_test_pred, label='Predicted', linewidth=2, alpha=0.8, linestyle='--')
plt.legend(fontsize=12)
plt.title(f'Best Model ({best_model}) Forecast on Test Set', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Target', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("artifacts/figures/best_forecast_test.png", dpi=150, bbox_inches='tight')
plt.close()

print("График прогноза сохранен в artifacts/figures/best_forecast_test.png")

График прогноза сохранен в artifacts/figures/best_forecast_test.png


In [20]:
experiments = [
    {
        "experiment_id": "B1",
        "task": "forecasting",
        "dataset": "S12-hw-dataset.csv",
        "seed": seed,
        "split_summary": "70/15/15 temporal",
        "window_size": None,
        "horizon": 1,
        "model_summary": "naive-last",
        "features_summary": "lag_1",
        "scaler": "None",
        "optimizer": "None",
        "lr": None,
        "epochs_trained": 0,
        "best_val_mae": b1_mae,
        "best_val_rmse": b1_rmse,
        "best_val_mape": b1_mape,
        "test_mae": "",
        "test_rmse": "",
        "test_mape": "",
        "notes": "baseline"
    },
       {
        "experiment_id": "B2",
        "task": "forecasting",
        "dataset": "S12-hw-dataset.csv",
        "seed": seed,
        "split_summary": "70/15/15 temporal",
        "window_size": 7,
        "horizon": 1,
        "model_summary": "moving-average",
        "features_summary": "rolling_mean_7",
        "scaler": "None",
        "optimizer": "None",
        "lr": None,
        "epochs_trained": 0,
        "best_val_mae": b2_mae,
        "best_val_rmse": b2_rmse,
        "best_val_mape": b2_mape,
        "test_mae": "",
        "test_rmse": "",
        "test_mape": "",
        "notes": "baseline"
    },
       {
        "experiment_id": "B3",
        "task": "forecasting",
        "dataset": "S12-hw-dataset.csv",
        "seed": seed,
        "split_summary": "70/15/15 temporal",
        "window_size": None,
        "horizon": 1,
        "model_summary": "Ridge",
        "features_summary": "lag_1,lag_7,lag_14,rolling_mean_7,rolling_std_7,day_of_week",
        "scaler": "StandardScaler",
        "optimizer": "None",
        "lr": None,
        "epochs_trained": 0,
        "best_val_mae": b3_val_mae,
        "best_val_rmse": b3_val_rmse,
        "best_val_mape": b3_val_mape,
        "test_mae": b3_test_mae,
        "test_rmse": b3_test_rmse,
        "test_mape": b3_test_mape,
        "notes": "baseline"
    },
        {
        "experiment_id": "R1",
        "task": "forecasting",
        "dataset": "S12-hw-dataset.csv",
        "seed": seed,
        "split_summary": "70/15/15 temporal",
        "window_size": WINDOW_SIZE,
        "horizon": 1,
        "model_summary": f"GRU(hidden_size=32)",
        "features_summary": "raw_sequence",
        "scaler": "None",
        "optimizer": "Adam",
        "lr": 1e-3,
        "epochs_trained": EPOCHS,
        "best_val_mae": best_val_mae,
        "best_val_rmse": best_val_rmse,
        "best_val_mape": best_val_mape,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_mape": test_mape,
        "notes": f"best_model_{best_model}"
    }
]
if os.path.exists(RUNS_PATH):
    os.remove(RUNS_PATH)

for exp in experiments:
    log_experiment(exp)

print("Все эксперименты сохранены в artifacts/runs.csv")
print(pd.read_csv(RUNS_PATH).to_string())

Все эксперименты сохранены в artifacts/runs.csv
  experiment_id         task             dataset  seed      split_summary  window_size  horizon        model_summary                                             features_summary          scaler optimizer     lr  epochs_trained  best_val_mae  best_val_rmse  best_val_mape  test_mae  test_rmse  test_mape             notes
0            B1  forecasting  S12-hw-dataset.csv    42  70/15/15 temporal          NaN        1           naive-last                                                        lag_1             NaN       NaN    NaN               0      6.440620       8.203475       4.392184       NaN        NaN        NaN          baseline
1            B2  forecasting  S12-hw-dataset.csv    42  70/15/15 temporal          7.0        1       moving-average                                               rolling_mean_7             NaN       NaN    NaN               0     12.726609      15.242137       8.830313       NaN        NaN        NaN        

In [21]:
print("ИТОГОВАЯ СВОДКА")
print(f"Лучшая модель: {best_model}")
print(f"Best Val MAE: {min(b1_mae, b2_mae, b3_val_mae, best_val_mae):.4f}")
print(f"Test MAE: {test_mae:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAPE: {test_mape:.4f}%")

ИТОГОВАЯ СВОДКА
Лучшая модель: Ridge
Best Val MAE: 6.4406
Test MAE: 7.3257
Test RMSE: 9.0340
Test MAPE: 4.6913%
